# Import Libraries

In [1]:
import sqlite3
import pandas as pd

# Connecting to Database and importing the data into Pandas

In [2]:
movies_conn = sqlite3.connect("db/movies.db")
ratings_conn = sqlite3.connect("db/ratings.db")

movies_df  = pd.read_sql_query("SELECT * FROM movies",  movies_conn)
ratings_df = pd.read_sql_query("SELECT * FROM ratings", ratings_conn)

movies_conn.close()
ratings_conn.close()


# Checking the shape of the Data (Rows and Columns)

In [3]:
print(f"  movies_df : {movies_df.shape[0]:,} rows × {movies_df.shape[1]} cols")
print(f"  ratings_df: {ratings_df.shape[0]:,} rows × {ratings_df.shape[1]} cols")

  movies_df : 45,430 rows × 12 cols
  ratings_df: 100,004 rows × 5 cols


# Analyzing Movies Data and preprocessing it

### 1. Visulizing the Data

In [4]:
movies_df.head(5)

,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,language,genres,status
0,2,tt0094675,Ariel,Taisto Kasurinen is a Finnish coal miner whose...,"[{""name"": ""Villealfa Filmproduction Oy"", ""id"":...",1988-10-21,0,0,69.0,None,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 80, ""name...",Released
1,3,tt0092149,Shadows in Paradise,"An episode in the life of Nikander, a garbage ...","[{""name"": ""Villealfa Filmproduction Oy"", ""id"":...",1986-10-16,0,0,76.0,None,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",Released
2,5,tt0113101,Four Rooms,It's Ted the Bellhop's first night on the job....,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...",1995-12-09,4000000,4300000,98.0,None,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...",Released
3,6,tt0107286,Judgment Night,"While racing to a boxing match, Frank, Mike, J...","[{""name"": ""Universal Pictures"", ""id"": 33}, {""n...",1993-10-15,0,12136938,110.0,None,"[{""id"": 28, ""name"": ""Action""}, {""id"": 53, ""nam...",Released
4,11,tt0076759,Star Wars,Princess Leia is captured and held hostage by ...,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...",1977-05-25,11000000,775398007,121.0,None,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",Released


### 2. Checking for Null Values across all columns

In [5]:
movies_df.isnull().sum()

movieId                    0
imdbId                     0
title                      0
overview                   0
productionCompanies        0
releaseDate                0
budget                     0
revenue                    0
runtime                    0
language               45430
genres                     0
status                     0
dtype: int64

### Since there are 45430 nulls in languages column we can drop the entire column as it's redudent data

In [6]:
movies_df = movies_df.drop(columns=["language"], errors="ignore")
movies_df.head(5)

,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,genres,status
0,2,tt0094675,Ariel,Taisto Kasurinen is a Finnish coal miner whose...,"[{""name"": ""Villealfa Filmproduction Oy"", ""id"":...",1988-10-21,0,0,69.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 80, ""name...",Released
1,3,tt0092149,Shadows in Paradise,"An episode in the life of Nikander, a garbage ...","[{""name"": ""Villealfa Filmproduction Oy"", ""id"":...",1986-10-16,0,0,76.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",Released
2,5,tt0113101,Four Rooms,It's Ted the Bellhop's first night on the job....,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...",1995-12-09,4000000,4300000,98.0,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...",Released
3,6,tt0107286,Judgment Night,"While racing to a boxing match, Frank, Mike, J...","[{""name"": ""Universal Pictures"", ""id"": 33}, {""n...",1993-10-15,0,12136938,110.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 53, ""nam...",Released
4,11,tt0076759,Star Wars,Princess Leia is captured and held hostage by ...,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...",1977-05-25,11000000,775398007,121.0,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",Released


### 3. Checking for duplicate values across important columns that are supposed to be unique identifiers

In [7]:
print('Duplicate movieID: ',movies_df.duplicated(subset=["movieId"]).sum())
print('Duplicate imdbId: ',movies_df.duplicated(subset=["imdbId"]).sum())

Duplicate movieID:  0
Duplicate imdbId:  16


### Visualizing the data with duplicate imdbId Values 

In [8]:
movies_df[movies_df.duplicated(subset=["imdbId"], keep=False)]

,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,genres,status
7018,15257,,Hulk vs. Wolverine,Department H sends in Wolverine to track down ...,"[{""name"": ""Marvel Studios"", ""id"": 420}]",2009-01-27,0,0,38.0,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 28, ""...",Released
12273,28500,,Before The Dinosaurs - Walking With Monsters,Many people think of the dinosaurs as the firs...,"[{""name"": ""BBC"", ""id"": 5996}]",2005-11-05,0,0,87.0,"[{""id"": 99, ""name"": ""Documentary""}, {""id"": 16,...",Released
12860,30146,,Gunbuster vs Diebuster Aim for the Top! The GA...,This double feature comprises of Gunbuster and...,"[{""name"": ""Bandai Visual Company"", ""id"": 528},...",2006-10-01,0,0,180.0,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 12, ""...",Released
14865,36337,,Delusion,"In this fast-paced, noirish road movie, a comp...","[{""name"": ""Cineville"", ""id"": 2832}]",1991-06-07,1000000,0,100.0,"[{""id"": 80, ""name"": ""Crime""}]",Released
14977,36663,,Dreamkiller,"A team of doctors experiment with a new, highl...",[],,2500000,0,110.0,"[{""id"": 9648, ""name"": ""Mystery""}, {""id"": 53, ""...",Released
18791,45514,,8,8 shorts centered around 8 themes directed by ...,[],2008-10-23,0,0,100.0,[],Rumored
19312,47116,,The Winner,Tou former boxers meet in the ring again after...,[],1979-03-09,0,0,78.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10749, ""n...",Rumored
21686,55576,,Last Stand at Saber River,"As America recovers from the Civil War, one ma...","[{""name"": ""Turner Network Television (TNT)"", ""...",1997-01-19,0,0,0.0,"[{""id"": 37, ""name"": ""Western""}, {""id"": 28, ""na...",Released
23882,65256,,H2O Just Add Water - The Movie,When three normal teenage girls stumble upon a...,"[{""name"": ""ZDF Enterprises"", ""id"": 5747}]",2011-10-10,0,0,91.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 12, ""na...",Released
25573,75015,,How I Unleashed World War II Part III: Among F...,How I Unleashed World War II tells the story o...,[],1970-04-06,0,0,73.0,"[{""id"": 10769, ""name"": ""Foreign""}, {""id"": 28, ...",Released


### We can see that more imdbId values are and empty string which passed the initial null test, also we can see that the productionCompanies and genres have [] empty array some have empty dates and empty runtimes and 0 budgets and revenue.

# Now let's clean Movies data 

## Note: since we only have to use 50 to 100 movies for the task, let's directly remove the imperfect data or noise.

### Movie ID

In [9]:
print(f"Empty imdbId rows : {(movies_df['movieId'] == '').sum():,}")
print(f"Total rows        : {len(movies_df):,}")

Empty imdbId rows : 0
Total rows        : 45,430


### imdbId

In [10]:
print(f"Empty imdbId rows : {(movies_df['imdbId'] == '').sum():,}")
print(f"Total rows        : {len(movies_df):,}")
movies_df = movies_df[movies_df["imdbId"] != ""].reset_index(drop=True)
print(f"Total rows        : {len(movies_df):,}")

Empty imdbId rows : 17
Total rows        : 45,430
Total rows        : 45,413


### Title

In [11]:
print(f"Empty imdbId rows : {(movies_df['title'] == '').sum():,}")
print(f"Total rows        : {len(movies_df):,}")
movies_df = movies_df[movies_df["title"] != ""].reset_index(drop=True)
print(f"Total rows        : {len(movies_df):,}")

Empty imdbId rows : 0
Total rows        : 45,413
Total rows        : 45,413


In [12]:
print(f"Empty imdbId rows : {(movies_df['budget'] == 0).sum():,}")
print(f"Total rows        : {len(movies_df):,}")
movies_df = movies_df[movies_df["budget"] != 0].reset_index(drop=True)
print(f"Total rows        : {len(movies_df):,}")

Empty imdbId rows : 36,535
Total rows        : 45,413
Total rows        : 8,878


In [13]:
print(f"Empty imdbId rows : {(movies_df['revenue'] == 0).sum():,}")
print(f"Total rows        : {len(movies_df):,}")
movies_df = movies_df[movies_df["revenue"] != 0].reset_index(drop=True)
print(f"Total rows        : {len(movies_df):,}")

Empty imdbId rows : 3,503
Total rows        : 8,878
Total rows        : 5,375


### overview

In [14]:
print(f"Empty overview rows : {(movies_df['overview'] == '').sum():,}")
movies_df = movies_df[movies_df["overview"] != ""].reset_index(drop=True)
print(f"Rows remaining: {len(movies_df):,}")

Empty overview rows : 11
Rows remaining: 5,364


### productionCompanies

In [15]:
print(f"Empty productionCompanies rows : {(movies_df['productionCompanies'] == '[]').sum():,}")
print(f"Total rows                     : {len(movies_df):,}")
movies_df = movies_df[movies_df["productionCompanies"] != "[]"].reset_index(drop=True)
print(f"Rows remaining: {len(movies_df):,}")

Empty productionCompanies rows : 177
Total rows                     : 5,364
Rows remaining: 5,187


### releaseDate

In [16]:
print(f"Empty releaseDate rows : {(movies_df['releaseDate'] == '').sum():,}")
print(f"Total rows             : {len(movies_df):,}")
movies_df = movies_df[movies_df["releaseDate"] != ""].reset_index(drop=True)
print(f"Rows remaining: {len(movies_df):,}")

Empty releaseDate rows : 0
Total rows             : 5,187
Rows remaining: 5,187


### runtime

In [17]:
print(f"Zero runtime rows  : {(movies_df['runtime'] == 0).sum():,}")
print(f"Empty runtime rows : {(movies_df['runtime'] == '').sum():,}")
print(f"Total rows         : {len(movies_df):,}")
movies_df = movies_df[(movies_df['runtime'] != 0) & (movies_df['runtime'] != '')].reset_index(drop=True)
print(f"Rows remaining: {len(movies_df):,}")

Zero runtime rows  : 0
Empty runtime rows : 0
Total rows         : 5,187
Rows remaining: 5,187


### genres

In [18]:
print(f"Empty genres rows : {(movies_df['genres'] == '[]').sum():,}")
print(f"Total rows        : {len(movies_df):,}")
movies_df = movies_df[movies_df["genres"] != "[]"].reset_index(drop=True)
print(f"Rows remaining: {len(movies_df):,}")

Empty genres rows : 1
Total rows        : 5,187
Rows remaining: 5,186


### status

In [19]:
print(movies_df["status"].value_counts())
print(f"\nNon-Released rows : {(movies_df['status'] != 'Released').sum():,}")
print(f"Total rows        : {len(movies_df):,}")
movies_df = movies_df[movies_df["status"] == "Released"].reset_index(drop=True)
print(f"Rows remaining: {len(movies_df):,}")

status
Released           5184
Rumored               1
Post Production       1
Name: count, dtype: int64

Non-Released rows : 2
Total rows        : 5,186
Rows remaining: 5,184


# Now let's Analyze Ratings Data and Preprocess it as well

### 1. Basic Analysis

In [20]:
print(f"  Shape          : {ratings_df.shape}")
print(f"  Unique users   : {ratings_df['userId'].nunique():,}")
print(f"  Unique movies  : {ratings_df['movieId'].nunique():,}")
print(f"  Rating range   : {ratings_df['rating'].min()} – {ratings_df['rating'].max()}")
print(f"  Avg rating     : {ratings_df['rating'].mean():.2f}")
ratings_df.head()

  Shape          : (100004, 5)
  Unique users   : 671
  Unique movies  : 9,066
  Rating range   : 0.5 – 5.0
  Avg rating     : 3.54


,ratingId,userId,movieId,rating,timestamp
0,1,1,31,2.5,1260759144
1,2,1,1029,3.0,1260759179
2,3,1,1061,3.0,1260759182
3,4,1,1129,2.0,1260759185
4,5,1,1172,4.0,1260759205


### 2. Checking for null values

In [21]:
print(ratings_df.isnull().sum())

ratingId     0
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### 3. checking for empty strings 

In [22]:
str_cols = ratings_df.select_dtypes(include="object").columns
if len(str_cols) == 0:
    print("  No string columns found in ratings_df")
else:
    for col in str_cols:
        empty = (ratings_df[col].str.strip() == "").sum()
        print(f"  {col:<20} : {empty:,}")

  No string columns found in ratings_df


### 4. Checking for Duplicate rows and redundent data.

In [23]:
print(ratings_df.duplicated().sum())

0


### 5. Checking for same user rating same movie twice

In [24]:
print(ratings_df.duplicated(subset=["userId", "movieId"]).sum())

0


# The Data for this Table is very clean and so let's also perform some EDA on it

### 1. Rating's count and Distribution

In [25]:
print("\n── Rating Value Counts ──")
print(ratings_df["rating"].value_counts().sort_index())


── Rating Value Counts ──
rating
0.5     1101
1.0     3326
1.5     1687
2.0     7271
2.5     4449
3.0    20064
3.5    10538
4.0    28750
4.5     7723
5.0    15095
Name: count, dtype: int64


In [26]:
ratings_per_user = ratings_df.groupby("userId")["rating"].count()
print("\n── Ratings per User ──")
print(f"  Min    : {ratings_per_user.min()}")
print(f"  Max    : {ratings_per_user.max():,}")
print(f"  Mean   : {ratings_per_user.mean():.1f}")
print(f"  Median : {ratings_per_user.median():.1f}")
print(f"  Users with < 5 ratings  : {(ratings_per_user < 5).sum():,}")
print(f"  Users with > 100 ratings: {(ratings_per_user > 100).sum():,}")


── Ratings per User ──
  Min    : 20
  Max    : 2,391
  Mean   : 149.0
  Median : 71.0
  Users with < 5 ratings  : 0
  Users with > 100 ratings: 258


In [27]:
ratings_per_movie = ratings_df.groupby("movieId")["rating"].count()
print("\n── Ratings per Movie ──")
print(f"  Min    : {ratings_per_movie.min()}")
print(f"  Max    : {ratings_per_movie.max():,}")
print(f"  Mean   : {ratings_per_movie.mean():.1f}")
print(f"  Median : {ratings_per_movie.median():.1f}")
print(f"  Movies with < 5 ratings : {(ratings_per_movie < 5).sum():,}")
print(f"  Movies with > 100 ratings: {(ratings_per_movie > 100).sum():,}")


── Ratings per Movie ──
  Min    : 1
  Max    : 341
  Mean   : 11.0
  Median : 3.0
  Movies with < 5 ratings : 5,570
  Movies with > 100 ratings: 149


### Let's remove the outlires using normal distribution 

In [28]:
ratings_per_movie = ratings_df.groupby("movieId")["rating"].count()

mean   = ratings_per_movie.mean()
std    = ratings_per_movie.std()
lower  = mean - std
lower = max(lower, 1)
upper  = mean + std

print(f"  Mean   : {mean:.1f}")
print(f"  Std    : {std:.1f}")
print(f"  Lower  : {lower:.1f}  (mean - 1std)")
print(f"  Upper  : {upper:.1f}  (mean + 1std)")

normal_movies = ratings_per_movie[(ratings_per_movie >= lower) & 
                                   (ratings_per_movie <= upper)]

print(f"\n  Movies in normal band : {len(normal_movies):,}")
print(f"  Movies outside band   : {len(ratings_per_movie) - len(normal_movies):,}")

  Mean   : 11.0
  Std    : 24.1
  Lower  : 1.0  (mean - 1std)
  Upper  : 35.1  (mean + 1std)

  Movies in normal band : 8,370
  Movies outside band   : 696


# Now Let's check how many movies in the movies table are in common with this 8,370 movies

In [29]:
normal_movie_ids = normal_movies.index.tolist()

in_common = movies_df[movies_df["movieId"].isin(normal_movie_ids)]
not_in_movies = set(normal_movie_ids) - set(movies_df["movieId"])
not_in_ratings = set(movies_df["movieId"]) - set(normal_movie_ids)

print(f"  Normal band movies (ratings_df) : {len(normal_movie_ids):,}")
print(f"  Cleaned movies_df               : {len(movies_df):,}")
print(f"  In common                       : {len(in_common):,}")
print(f"  In ratings but not in movies_df : {len(not_in_movies):,}")
print(f"  In movies_df but not in ratings : {len(not_in_ratings):,}")
print(f"\n  Coverage: {len(in_common)/len(movies_df)*100:.1f}% of movies_df have ratings in normal band")

  Normal band movies (ratings_df) : 8,370
  Cleaned movies_df               : 5,184
  In common                       : 841
  In ratings but not in movies_df : 7,529
  In movies_df but not in ratings : 4,343

  Coverage: 16.2% of movies_df have ratings in normal band


# Since the Task require us to analyse 50 to 100 Movies we will go ahed and Create a Master Dataset with a sample of 100 from these 841 movies with perfect Data

In [30]:
sample_df = in_common.sample(n=100, random_state=42).reset_index(drop=True)

print(f"Sample size : {len(sample_df)}")
display(sample_df.head(5))

Sample size : 100


,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,genres,status
0,2022,tt0280590,Mr. Deeds,"When Longfellow Deeds, a small-town pizzeria o...","[{""name"": ""New Line Cinema"", ""id"": 12}, {""name...",2002-06-28,50000000,171269535,96.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",Released
1,27022,tt0963966,The Sorcerer's Apprentice,Balthazar Blake is a master sorcerer in modern...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",2010-07-13,150000000,215283742,109.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 12, ""na...",Released
2,88,tt0092890,Dirty Dancing,Expecting the usual tedium that accompanies a ...,"[{""name"": ""Great American Films Limited Partne...",1987-08-21,6000000,213954274,100.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10402, ""n...",Released
3,2486,tt0449010,Eragon,"In his homeland of Alagaesia, a farm boy happe...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...",2006-12-14,100000000,249288105,104.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""na...",Released
4,248,tt0055312,Pocketful of Miracles,"Damon Runyon's fairytale, sweet and funny, is ...","[{""name"": ""Franton Production"", ""id"": 1044}]",1961-12-18,2900000,5000000,136.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",Released


# Now Let's Join User Ratings as aggrigate columns to these 100 movies and also join the user ids and ratings as json dumps

In [31]:
import json

# ── 1. Aggregate ratings per movie ───────────────────────────────
movie_ratings = ratings_df.groupby("movieId").agg(
    avg_rating    = ("rating", "mean"),
    rating_count  = ("rating", "count")
).reset_index()

# ── 2. Add user_ratings JSON column ──────────────────────────────
user_ratings = (
    ratings_df.groupby("movieId")
    .apply(lambda x: [{"userId": int(r.userId), "rating": float(r.rating)} 
                      for r in x.itertuples()], include_groups=False)
    .reset_index()
    .rename(columns={0: "user_ratings"})
)
user_ratings["user_ratings"] = user_ratings["user_ratings"].apply(json.dumps)

# ── 3. Merge both into movie_ratings ─────────────────────────────
movie_ratings = movie_ratings.merge(user_ratings, on="movieId", how="left")

# ── 4. Join with sample_df ────────────────────────────────────────
sample_df = sample_df.merge(movie_ratings, on="movieId", how="left")

# ── 5. Verify ─────────────────────────────────────────────────────
print(f"Shape after join  : {sample_df.shape}")
print(f"Null avg_rating   : {sample_df['avg_rating'].isnull().sum()}")
print(f"Null rating_count : {sample_df['rating_count'].isnull().sum()}")
print(f"Null user_ratings : {sample_df['user_ratings'].isnull().sum()}")
sample_df.head(3)

Shape after join  : (100, 14)
Null avg_rating   : 0
Null rating_count : 0
Null user_ratings : 0


,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,genres,status,avg_rating,rating_count,user_ratings
0,2022,tt0280590,Mr. Deeds,"When Longfellow Deeds, a small-town pizzeria o...","[{""name"": ""New Line Cinema"", ""id"": 12}, {""name...",2002-06-28,50000000,171269535,96.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",Released,3.423077,13,"[{""userId"": 23, ""rating"": 4.0}, {""userId"": 232..."
1,27022,tt0963966,The Sorcerer's Apprentice,Balthazar Blake is a master sorcerer in modern...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",2010-07-13,150000000,215283742,109.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 12, ""na...",Released,4.333333,3,"[{""userId"": 73, ""rating"": 4.5}, {""userId"": 332..."
2,88,tt0092890,Dirty Dancing,Expecting the usual tedium that accompanies a ...,"[{""name"": ""Great American Films Limited Partne...",1987-08-21,6000000,213954274,100.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10402, ""n...",Released,2.891304,23,"[{""userId"": 33, ""rating"": 3.0}, {""userId"": 67,..."


# Storing Final Data into a DB file

In [32]:
conn = sqlite3.connect("db/Final_movies.db")

sample_df.to_sql("Final_movies", conn, if_exists="replace", index=False)

conn.close()